In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import pyaging as pya
import torch
from art.estimators.regression.pytorch import PyTorchRegressor
import pathlib
from scipy.stats import iqr
from art.attacks.evasion import BasicIterativeMethod
from metrics import get_reg_metrics
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
path = "D:/bioTest/attack/"
pheno = pd.read_excel(f"{path}/data/controls.xlsx", index_col=0)
betas = pd.read_pickle(f"{path}data/betas.pkl")

feats_pheno = ['Age', 'Sex', 'Tissue']
pheno = pheno[feats_pheno]

df_clocks = pd.merge(pheno, betas, left_index=True, right_index=True)
df_clocks['Female'] = (df_clocks['Sex'] == 'F').astype(int)

In [78]:
clocks_names = ["altumage"]

for clocks in clocks_names:

    path_clock = f"{path}/{clocks}"
    pathlib.Path(f"{path_clock}").mkdir(parents=True, exist_ok=True)

    dir = "E:/YandexDisk/pydnameth/datasets/pyaging"

    adata = pya.pp.df_to_adata(df_clocks, metadata_cols=['Sex', 'Tissue'], imputer_strategy='knn', verbose=True)
    pya.pred.predict_age(adata=adata, dir=dir, clock_names=clocks, verbose=True)
    results = pd.merge(pheno.loc[:, feats_pheno], adata.obs[clocks], left_index=True, right_index=True)

    logger = pya.logger.Logger('test_logger')
    device = 'cpu'
    indent_level = 1

    clock = pya.pred.load_clock(clocks, device, dir, logger, indent_level=indent_level)
    clock_features = clock.features
    clock_reference_values = clock.reference_values

    common_cpgs = list(set(clock_features).intersection(betas.columns))
    differ_cpgs = list(set(clock_features).difference(betas.columns))
    missing_indices = [clock_features.index(curr_cpg) for curr_cpg in differ_cpgs]
    if clock_reference_values is not None:
        missing_references = [clock_reference_values[ref] for ref in missing_indices]
    else:
        mean_df = np.mean(betas.loc[:, common_cpgs])
        missing_references = [mean_df for ref in missing_indices]
    df = betas.loc[:, common_cpgs]
    df[differ_cpgs] = pd.DataFrame([missing_references], index=df.index)
    df = pd.merge(df, results, left_index=True, right_index=True)

    feats = clock_features
    ids_feat = list(range(len(feats)))
    col_trgt = 'Age'
    col_pred = clocks

    df["Error"] = df[col_pred] - df[col_trgt]
    df["|Error|"] = df["Error"].abs()
    df['Data'] = 'Real'
    df['Eps'] = 'Origin'

    model = clock
    for model_component in model._modules:
        if isinstance(getattr(model, model_component), pya.models._base_models.LinearModel):
            getattr(model, model_component).linear.weight = torch.nn.parameter.Parameter(getattr(model, model_component).linear.weight.to(torch.float32))
            getattr(model, model_component).linear.bias = torch.nn.parameter.Parameter(getattr(model, model_component).linear.bias.to(torch.float32))
        if isinstance(getattr(model, model_component), pya.models._base_models.PCLinearModel):
            getattr(model, model_component).linear.weight = torch.nn.parameter.Parameter(getattr(model, model_component).linear.weight.to(torch.float32))
            getattr(model, model_component).linear.bias = torch.nn.parameter.Parameter(getattr(model, model_component).linear.bias.to(torch.float32))
            getattr(model, model_component).center = torch.nn.parameter.Parameter(getattr(model, model_component).center.to(torch.float32))
            getattr(model, model_component).rotation = torch.nn.parameter.Parameter(getattr(model, model_component).rotation.to(torch.float32))
        if isinstance(getattr(model, model_component), pya.models._base_models.AltumAgeNeuralNetwork):
            for curr_model_component in model.base_model._modules:
                if isinstance(getattr(model.base_model, curr_model_component), torch.nn.Linear):
                    getattr(model.base_model, curr_model_component).weight = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).weight.to(torch.float32))
                    getattr(model.base_model, curr_model_component).bias = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).bias.to(torch.float32))
                if isinstance(getattr(model.base_model, curr_model_component), torch.nn.BatchNorm1d):
                    getattr(model.base_model, curr_model_component).weight = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).weight.to(torch.float32))
                    getattr(model.base_model, curr_model_component).bias = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).bias.to(torch.float32))
                    getattr(model.base_model, curr_model_component).running_mean = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).running_mean.to(torch.float32))
                    getattr(model.base_model, curr_model_component).running_mean.requires_grad = False
                    getattr(model.base_model, curr_model_component).running_var = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).running_var.to(torch.float32))
                    getattr(model.base_model, curr_model_component).running_var.requires_grad = False

    art_regressor = PyTorchRegressor(
        model=model,
        loss=torch.nn.L1Loss(),
        input_shape=[len(feats)],
        use_amp=False,
        opt_level="O1",
        loss_scale="dynamic",
        channels_first=True,
        clip_values=None,
        preprocessing_defences=None,
        postprocessing_defences=None,
        preprocessing=(0.0, 1.0),
        device_type="cpu",
    )

    epsilons = sorted(list(set.union(
        set(np.linspace(0.1, 1.0, 10)), 
        set(np.linspace(0.01, 0.1, 10)),
    )))
    df_eps = pd.DataFrame(index=epsilons)

    for eps_raw in epsilons:

        eps = np.array([eps_raw * iqr(df.loc[:, feat].values) for feat in feats])
        eps_step = np.array([0.2 * eps_raw * iqr(df.loc[:, feat].values) + 1e-6 for feat in feats])

        attacks = {
            'BasicIterative': BasicIterativeMethod(
                estimator=art_regressor,
                eps=eps,
                eps_step=eps_step,
                max_iter=100,
                targeted=False,
                batch_size=512,
                verbose=True
            )
        }

        for attack_name, attack in attacks.items():
            path_curr = f"{path_clock}/Evasion/{attack_name}/eps_{eps_raw:0.4f}"
            pathlib.Path(f"{path_curr}").mkdir(parents=True, exist_ok=True)

            X_adv = attack.generate(df.loc[:, feats].values.astype(np.float32))
            
            df_adv = df.loc[:, [col_trgt]].copy()
            df_adv.loc[:, feats] = X_adv
            df_adv[col_pred] = model(torch.from_numpy(np.float32(df_adv.loc[:, feats].values))).cpu().detach().numpy().ravel()
            df_adv["Error"] = df_adv[col_pred] - df_adv[col_trgt]
            df_adv["abs(Error)"] = df_adv["Error"].abs()
            df_adv.loc[:, "Error Origin"] = df.loc[:, col_pred] - df.loc[:, col_trgt]
            df_adv.loc[:, "Error Attack"] = df_adv.loc[:, col_pred] - df_adv.loc[:, col_trgt]
            df_adv['Error Diff'] = df_adv['Error Attack'] - df_adv['Error Origin']
            df_adv['abs(Error Diff)'] = df_adv['Error Diff'].abs()
                
            #df_adv.to_excel(f"{path_curr}/df.xlsx", index_label='sample_id')

            metrics = get_reg_metrics()
            metrics_cols = [f"{m}" for m in metrics]
            df_metrics = pd.DataFrame(index=metrics_cols)
            for m in metrics:
                m_val = float(metrics[m][0](torch.from_numpy(np.float32(df.loc[:, col_pred].values)), torch.from_numpy(np.float32(df.loc[:, col_trgt].values))).numpy())
                df_metrics.at[f"{m}", 'Origin'] = m_val
                metrics[m][0].reset()
                m_val = float(metrics[m][0](torch.from_numpy(np.float32(df_adv.loc[:, col_pred].values)), torch.from_numpy(np.float32(df.loc[:, col_trgt].values))).numpy())
                df_metrics.at[f"{m}", 'Attack'] = m_val
                metrics[m][0].reset()
            df_metrics.to_excel(f"{path_curr}/metrics.xlsx", index_label='Metrics')
        
            df_eps.loc[eps_raw, f"Origin_MAE"] = df_metrics.at[f'mean_absolute_error', 'Origin']
            df_eps.loc[eps_raw, f"{attack_name}_MAE"] = df_metrics.at[f'mean_absolute_error', 'Attack']

    df_eps.to_excel(f"{path_clock}/Evasion/df_eps.xlsx", index_label='eps')

    df_fig = df_eps.copy()
    df_fig['Eps'] = df_fig.index.values
    df_fig = df_fig.melt(id_vars="Eps", var_name='Method', value_name="MAE")
    fig = plt.figure()
    sns.set_theme(style='whitegrid', font_scale=1)
    lines = sns.lineplot(
        data=df_fig,
        x='Eps',
        y="MAE",
        hue=f"Method",
        style=f"Method",
        markers=True,
        dashes=False,
    )
    plt.xscale('log')
    lines.set_xlabel(r'$\epsilon$')
    x_min = 0.009
    x_max = 1.05
    mae_basic = df_eps.at[0.01, f"Origin_MAE"]
    lines.set_xlim(x_min, x_max)
    plt.gca().plot(
        [x_min, x_max],
        [mae_basic, mae_basic],
        color='k',
        linestyle='dashed',
        linewidth=1
    )
    plt.savefig(f"{path_clock}/Evasion/line_mae_vs_eps.png", bbox_inches='tight', dpi=200)
    plt.savefig(f"{path_clock}/Evasion/line_mae_vs_eps.pdf", bbox_inches='tight')
    plt.close(fig)

|-----> 🏗️ Starting df_to_adata function
|-----> ⚙️ Create anndata object started
|-----> ✅ Create anndata object finished [1.4632s]
|-----> ⚙️ Add metadata to anndata started
|-----------> Adding provided metadata to adata.obs
|-----> ✅ Add metadata to anndata finished [0.0011s]
|-----> ⚙️ Log data statistics started
|-----------> There are 729 observations
|-----------> There are 411976 features
|-----------> Total missing values: 0
|-----------> Percentage of missing values: 0.00%
|-----> ✅ Log data statistics finished [0.4288s]
|-----> ⚙️ Impute missing values started
|-----------> No missing values found. No imputation necessary
|-----> ✅ Impute missing values finished [0.4103s]
|-----> 🎉 Done! [2.8115s]
|-----> 🏗️ Starting predict_age function
|-----> ⚙️ Set PyTorch device started
|-----------> Using device: cpu
|-----> ✅ Set PyTorch device finished [0.0029s]
|-----> 🕒 Processing clock: altumage
|-----------> ⚙️ Load clock started
|-----------------> Data found in E:/YandexDisk/p

C:\Users\alena\AppData\Local\Temp\ipykernel_42536\1660118466.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[differ_cpgs] = pd.DataFrame([missing_references], index=df.index)
C:\Users\alena\AppData\Local\Temp\ipykernel_42536\1660118466.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[differ_cpgs] = pd.DataFrame([missing_references], index=df.index)
C:\Users\alena\AppData\Local\Temp\ipykernel_42536\1660118466.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.in

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_42536\1660118466.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

...

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_42536\1660118466.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_42536\1660118466.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
C:\Users\alena\AppData\Local\Temp\ipykernel_42536\1660118466.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=